# 04 · Volatility connectedness: stablecoins vs crypto and traditional markets

Daily Parkinson volatility $V_t = |\ln(H_t/L_t)| / \sqrt{4\ln 2}$ for USDT, USDC, BTC, ETH, DXY, gold,
S&P 500 and the KBW bank index (VIX enters the regression as a level), Jan 2020 – Feb 2026, weekdays only.

1. **Table I**: descriptive statistics and stablecoin-volatility regressions (Newey–West, year FE).
2. **Table II**: Diebold–Yilmaz connectedness from a static VAR and from a TVP-VAR (Koop–Korobilis forgetting
   factors λ = 0.99, κ = 0.96; generalised FEVD at H = 10).

> **Data.** The traditional-market series are Bloomberg-licensed and not distributed. Supply
> `data/external/traditional_market_data.xlsx` (format in `data/README.md`) to run this notebook.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import spillover as sp, plotting, regressions
from stablecoin_risk.data import loaders

vol = sp.build_volatility_panel(loaders.load_daily_crypto(), loaders.load_traditional_markets())
print(vol.shape, vol.index.min().date(), "->", vol.index.max().date())
sp.descriptive_statistics(vol)

(1603, 9) 2020-01-01 -> 2026-02-20


,N,Mean,SD,ADF,JB,Q(10)
USDT,1603,0.0039,0.0096,-8.92***,289604***,3159***
USDC,1603,0.0023,0.0074,-4.65***,334678***,2470***
BTC,1603,0.0314,0.0378,-11.61***,32174676***,140***
ETH,1603,0.0397,0.0285,-5.03***,109591***,1129***
DXY,1603,0.0037,0.0019,-6.98***,3626***,1932***
XAU,1603,0.009,0.0057,-9.36***,102076***,2008***
S&P500,1603,0.0075,0.0055,-6.42***,23754***,4563***
KBW,1603,0.0127,0.0084,-6.97***,29994***,3544***
VIX,1603,21.1,7.815,-4.72***,10047***,11747***


In [3]:
regressions.results_table({c: sp.stablecoin_volatility_regression(vol, c) for c in ("USDT", "USDC")})

,USDT,USDC
const,0.00114*,-0.000316
(const),(0.000689),(0.000519)
BTC,-0.00347,-0.00727
(BTC),(0.00323),(0.00579)
ETH,0.0318***,0.0291***
(ETH),(0.00721),(0.00585)
DXY,0.074,0.206*
(DXY),(0.112),(0.121)
S&P500,-0.0474,-0.0495
(S&P500),(0.0388),(0.0639)


Stablecoin volatility is highly persistent (own lag ≈ 0.57) and loads on ETH volatility, not BTC, once both are
included; macro factors add little. Volatility is stationary (ADF) and fat-tailed (JB), which motivates the
connectedness approach below.

In [4]:
data = vol[sp.SPILLOVER_ASSETS]
static = sp.static_spillover(data)
print(f"Static VAR({static.lags}) total spillover index: {static.tsi:.2f}%")
static.table.round(2)

Static VAR(2) total spillover index: 28.34%


,USDT,USDC,BTC,ETH,DXY,XAU,S&P500,KBW,FROM others
USDT,88.1,9.33,0.22,0.97,0.26,0.23,0.82,0.08,11.9
USDC,1.78,94.43,0.13,1.11,1,0.44,0.13,0.97,5.57
BTC,0.24,0.22,67.65,28.33,0.22,0.5,1.6,1.25,32.35
ETH,0.97,0.98,21.47,65.97,1.08,2.04,5.26,2.23,34.03
DXY,0.18,0.38,0.22,1.31,67.4,6.63,15.9,7.98,32.6
XAU,0.02,0.24,0.69,3.2,6.94,74.99,7.14,6.78,25.01
S&P500,0.06,0.1,1.26,4.19,5.69,3.55,57.8,27.35,42.2
KBW,0.02,0.33,1.16,3.15,3.21,4.1,31.12,56.93,43.07
TO others,3.26,11.57,25.15,42.26,18.39,17.49,61.96,46.64,NaN
NET,-8.64,6.01,-7.2,8.23,-14.21,-7.52,19.76,3.57,NaN


In [5]:
tvp = sp.tvp_var(data, lags=2)
avg, tsi_t, _ = sp.tvp_connectedness(tvp)
print(f"TVP-VAR(2) total spillover index: {avg.tsi:.2f}%")
avg.table.round(2)

TVP-VAR(2) total spillover index: 41.43%


,USDT,USDC,BTC,ETH,DXY,XAU,S&P500,KBW,FROM others
USDT,68.63,11.62,4.68,6.8,2.24,1.9,2.4,1.73,31.37
USDC,13.34,69.5,3.35,5.09,1.72,2.23,2.66,2.12,30.5
BTC,4.62,3.04,52.38,26.85,2.51,4.22,3.7,2.68,47.62
ETH,6.76,5.61,24.46,47.78,2.99,4.15,4.8,3.46,52.22
DXY,2.61,1.98,2.83,3.72,62.11,12.2,8.21,6.33,37.89
XAU,3.74,3.04,4,4.73,11.39,59.62,6.95,6.52,40.38
S&P500,3.01,4.66,3.7,4.49,5.99,4.93,54.9,18.32,45.1
KBW,3.54,2.49,3.26,3.75,5.84,5.44,22.02,53.66,46.34
TO others,37.62,32.44,46.29,55.43,32.67,35.05,50.73,41.17,NaN
NET,6.25,1.95,-1.34,3.21,-5.21,-5.32,5.63,-5.17,NaN


In [6]:
fig = plotting.plot_connectedness(tsi_t)

In [7]:
fig = plotting.plot_net_spillovers(avg.table)

Allowing parameters to vary raises total connectedness from 28% (static) to **41.4%**. Both stablecoins are **net
transmitters** of volatility (NET > 0), and they spill mostly into each other (USDT→USDC 13%, USDC→USDT 12%).
Stablecoin volatility is therefore not a passive by-product of crypto or macro risk.